# Bài tập buổi 7

**Sinh viên thực hiện:** Trần Tấn Thịnh, **MSSV:** 2510236

---

**Bài 1:** Dự đoán tỷ lệ sống sót Titanic bằng Logistic Regression.

**Bài 2:** Phân loại Dry Bean Dataset bằng Logistic Regression và KNN.

In [9]:
# Chuẩn bị môi trường và thư viện
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.io import arff

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

---
## Bài 1: Titanic Dataset

In [10]:
# 1. Tải dữ liệu và áp dụng các quy tắc tiền xử lý/chia tập như bài tập trước
df_titanic = sns.load_dataset('titanic')

# Loại bỏ các cột rò rỉ nhãn và dư thừa
leaky = ['alive', 'who', 'adult_male', 'class', 'deck', 'embark_town', 'alone']
df_titanic = df_titanic.drop(columns=leaky)

# Tách X, y
X = df_titanic.drop(columns=['survived'])
y = df_titanic['survived']

# Cùng cách chia dữ liệu train/test/val (70/15/15) có stratify
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=(15/85), stratify=y_tmp, random_state=42)

# Khởi tạo Pipeline tiền xử lý
num_cols = ['age', 'sibsp', 'parch', 'fare']
cat_cols = ['sex', 'embarked']
ord_cols = ['pclass']

pipe_num = Pipeline([
    ('imputer', SimpleImputer(strategy='median')), 
    ('scaler', RobustScaler())
])
pipe_cat = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')), 
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocess = ColumnTransformer([
    ('num', pipe_num, num_cols), 
    ('cat', pipe_cat, cat_cols), 
    ('ord', 'passthrough', ord_cols)
])

# Fit TRÊN TẬP TRAIN và transform
X_train_t = preprocess.fit_transform(X_train)
X_test_t = preprocess.transform(X_test)

In [11]:
# 2. Xây dựng và đánh giá mô hình Logistic Regression
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_t, y_train)
y_pred = log_reg.predict(X_test_t)

print('--- Đánh giá Kết quả Logistic Regression (Titanic) ---')
print(f'Accuracy:  {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred):.4f}')
print(f'F1-score:  {f1_score(y_test, y_pred):.4f}\n')

print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

--- Đánh giá Kết quả Logistic Regression (Titanic) ---
Accuracy:  0.7761
Precision: 0.7442
Recall:    0.6275
F1-score:  0.6809

Confusion Matrix:
[[72 11]
 [19 32]]


### Nhận xét & So sánh mô hình trên tập Titanic

- **Về Linear Regression (Bài tập trước):** Đây là thuật toán dành cho bài toán *Hồi quy* (dự đoán một giá trị liên tục). Nếu áp dụng Linear Regression vào dự đoán phân loại sống/chết (0 hoặc 1), đầu ra có thể nhỏ hơn 0 hoặc lớn hơn 1, dễ bị tác động mạnh bởi giá trị ngoại lai (outliers) và yêu cầu người dùng phải tự thiết lập một ngưỡng (threshold) phân loại cứng ngắc.
- **Về Logistic Regression:** Thuật toán này sinh ra dành cho bài toán *Phân loại* nhị phân. Thông qua hàm Sigmoid, nó ép mọi đầu ra về một xác suất mượt mà trong khoảng [0, 1]. Từ đó mô hình phân loại người sống sót một cách tự nhiên và chính xác hơn dựa trên toán học chuẩn.
- **Kết luận:** Đối với bài toán phân loại hành khách Titanic, mô hình **Logistic Regression phù hợp và có hiệu suất đáng tin cậy hơn hẳn** so với Linear Regression.

---
## Bài 2: Dry Bean Dataset

In [12]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Đọc trực tiếp từ 2 file đã được chia sẵn
df_train = pd.read_csv('Dry_Bean_Dataset/dry_bean_train.csv')
df_test = pd.read_csv('Dry_Bean_Dataset/dry_bean_test.csv')

# Tách X (đặc trưng) và y (nhãn) với tên cột chính xác là 'class'
X_b_train = df_train.drop(columns=['class'])
y_b_train = df_train['class']

X_b_test = df_test.drop(columns=['class'])
y_b_test = df_test['class']

# Mã hóa nhãn (Label Encoding) chuyển dạng chữ (SEKER, BARBUNYA...) sang số
le = LabelEncoder()
y_b_train_enc = le.fit_transform(y_b_train)
y_b_test_enc = le.transform(y_b_test)

# Chuẩn hóa dữ liệu với StandardScaler (Bắt buộc cho Logistic Regression và KNN)
scaler_b = StandardScaler()
X_b_train_sc = scaler_b.fit_transform(X_b_train)
X_b_test_sc = scaler_b.transform(X_b_test)

In [13]:
# 2. Xây dựng và đánh giá Mô hình Logistic Regression
log_reg_beans = LogisticRegression(max_iter=1000, random_state=42)
log_reg_beans.fit(X_b_train_sc, y_b_train_enc)
y_pred_log_beans = log_reg_beans.predict(X_b_test_sc)

print('--- Đánh giá Logistic Regression (Dry Bean) ---')
print(f'Accuracy: {accuracy_score(y_b_test_enc, y_pred_log_beans):.4f}')
print('\nClassification Report:')
print(classification_report(y_b_test_enc, y_pred_log_beans, target_names=le.classes_))

--- Đánh giá Logistic Regression (Dry Bean) ---
Accuracy: 0.9192

Classification Report:
              precision    recall  f1-score   support

    BARBUNYA       0.93      0.89      0.91       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.91      0.94      0.93       326
    DERMASON       0.93      0.91      0.92       709
       HOROZ       0.96      0.94      0.95       372
       SEKER       0.93      0.94      0.94       406
        SIRA       0.86      0.88      0.87       527

    accuracy                           0.92      2709
   macro avg       0.93      0.93      0.93      2709
weighted avg       0.92      0.92      0.92      2709



In [14]:
# 3. Xây dựng và đánh giá Mô hình K-Nearest Neighbors (KNN)
knn_beans = KNeighborsClassifier(n_neighbors=5)
knn_beans.fit(X_b_train_sc, y_b_train_enc)
y_pred_knn_beans = knn_beans.predict(X_b_test_sc)

print('--- Đánh giá K-Nearest Neighbors - KNN (Dry Bean) ---')
print(f'Accuracy: {accuracy_score(y_b_test_enc, y_pred_knn_beans):.4f}')
print('\nClassification Report:')
print(classification_report(y_b_test_enc, y_pred_knn_beans, target_names=le.classes_))

--- Đánh giá K-Nearest Neighbors - KNN (Dry Bean) ---
Accuracy: 0.9155

Classification Report:
              precision    recall  f1-score   support

    BARBUNYA       0.93      0.88      0.90       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.90      0.95      0.92       326
    DERMASON       0.92      0.91      0.91       709
       HOROZ       0.96      0.93      0.95       372
       SEKER       0.95      0.94      0.94       406
        SIRA       0.85      0.87      0.86       527

    accuracy                           0.92      2709
   macro avg       0.93      0.93      0.93      2709
weighted avg       0.92      0.92      0.92      2709

